##Environment setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/shimnet_workshop
!pip install nmrglue

##Display script used to collect calibration spectra

In [ ]:
!cat calibration_loop/sweep_shims_lineshape_Z1Z2.py

##Convolution - idea

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import os


file_path = "/content/drive/MyDrive/shimnet_workshop/data/spectrum_singlet_and_multiplet2.txt"
df = pd.read_csv(file_path, header=None, names=["intensity"])
df["index"] = range(1, len(df) + 1)
original_spectrum = torch.tensor(df["intensity"].values, dtype=torch.float32).view(1,1,-1)


response_function_length = 71
resp = torch.cat([torch.arange(response_function_length//2+1).float(), torch.zeros(response_function_length//2)])

# halfwidth = 7 #change from 1 to 15 only total values
# resp = torch.arange(1, 2*halfwidth+1, dtype=torch.float32)
# resp = torch.cat([torch.zeros(halfwidth), resp, torch.zeros(halfwidth-1)])
resp /= resp.sum()


distorted = F.conv1d(original_spectrum, resp.view(1,1,-1), padding=len(resp)//2)


fig = plt.figure(figsize=(14, 10))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.2])

ax1 = fig.add_subplot(gs[0,0])
ax1.plot(df["index"], df["intensity"], label="Ideal Spectrum (f)"); ax1.invert_xaxis(); ax1.grid(True)
ax1.legend()

ax2 = fig.add_subplot(gs[0,1])
ax2.plot(resp,label="Distortion (g)",); ax2.grid(True)
ax2.legend()

ax3 = fig.add_subplot(gs[1,:])
ax3.plot(original_spectrum[0,0], label="Ideal Spectrum (f)",  c="0.5")
ax3.plot(distorted[0,0].detach(), label="Distorted Spectrum (f*g)", color="red")
ax3.invert_xaxis()
ax3.legend()
ax3.grid(True)

plt.tight_layout(); plt.show()





## Function to fit the distortion function (determine g)


In [11]:
def fit_kernel(base, target, kernel_size, steps=12000, verbose=False):

    best_kernel = None
    best_loss = float('inf')

    kernel = torch.ones((1,1,kernel_size), dtype=base.dtype)
    kernel /= kernel_size
    kernel.requires_grad = True

    optimizer = torch.optim.Adam([kernel])

    for epoch in range(steps):
        spe_est = torch.conv1d(base, kernel, padding='same')
        loss = torch.mean(abs(target - spe_est)**2) #torch.nn.functional.mse_loss(spe_est, target)

        if loss < best_loss:
            best_loss = loss
            best_kernel = kernel.detach().clone()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if verbose and (epoch+1) % 200 == 0:
            print(epoch, loss.item())
    return best_kernel, best_loss.item()

## Fitting the distortion function (noiseless case)

In [ ]:
response_function = resp.reshape(1, 1, -1)
distorted_spectrum = distorted[0]
response_function_estimate_length = 71 # long: 91, exact: 71, short: 51

response_function_estimate, loss = fit_kernel(original_spectrum, distorted_spectrum, kernel_size=response_function_estimate_length, verbose=False)

plt.plot(response_function[0,0])
plt.plot(response_function_estimate[0,0], ls="--",label="estimate")
plt.title(f"Loss: {loss}")

spectrum_estimate = torch.conv1d(original_spectrum, response_function_estimate, padding='same')
plt.figure(figsize=(16,4))
plt.plot(original_spectrum[0,0], c="0.5", alpha=0.5,label="undistorted")
plt.plot(distorted_spectrum[0],label="distorted")
plt.plot(spectrum_estimate[0,0], ls="--",label="estimate")
plt.legend()


### Fitting the distortion function (noise present)

In [ ]:
noise_level = 0.001 # small: 0.001, medium: 0.01, large: 0.1
response_function_estimate_length = 71

response_function = resp.reshape(1, 1, -1)
distorted_spectrum_noised = distorted_spectrum + noise_level*torch.rand_like(distorted_spectrum)
original_spectrum_noised = original_spectrum + noise_level*torch.rand_like(original_spectrum)

response_function_estimate, loss = fit_kernel(original_spectrum_noised, distorted_spectrum_noised, kernel_size=response_function_estimate_length, verbose=False)

plt.plot(response_function[0,0])
plt.plot(response_function_estimate[0,0], ls="--")
plt.title(f"Loss: {loss}")

spectrum_estimate = torch.conv1d(original_spectrum_noised, response_function_estimate, padding='same')
plt.figure(figsize=(16,4))
plt.plot(original_spectrum_noised[0,0], c="0.5", alpha=0.5,label="undistorted")
plt.plot(distorted_spectrum_noised[0],label="distorted")
plt.plot(spectrum_estimate[0,0], ls="--",label="estimate")
plt.legend()

### Fitting the distortion function on part of the spectrum

In [ ]:
noise_level = 0.001 # small: 0.001, medium: 0.01
response_function_estimate_length = 71
fitting_range = (50, 190) # singlet: (50, 190), multiplet: (300, 450)

response_function = resp.reshape(1, 1, -1)
distorted_spectrum_noised = distorted_spectrum + noise_level*torch.rand_like(distorted_spectrum)
original_spectrum_noised = original_spectrum + noise_level*torch.rand_like(original_spectrum)

response_function_estimate, loss = fit_kernel(original_spectrum_noised[:,:,fitting_range[0]:fitting_range[1]], distorted_spectrum_noised[:,fitting_range[0]:fitting_range[1]], kernel_size=response_function_estimate_length, verbose=False)

plt.plot(response_function[0,0])
plt.plot(response_function_estimate[0,0], ls="--")
plt.title(f"Loss: {loss}")

spectrum_estimate = torch.conv1d(original_spectrum_noised, response_function_estimate, padding='same')
plt.figure(figsize=(16,4))
plt.plot(original_spectrum_noised[0,0], c="0.5", alpha=0.5,label="undistorted")
plt.plot(distorted_spectrum_noised[0],label="distorted")
plt.plot(spectrum_estimate[0,0], ls="--",label="estimate")
plt.legend()

## Shim Coil Response Function (SCRF) extraction from the calibration data

In [ ]:
from extract_scrf_from_fids import run
import os

data_dir = "./SCRF_extraction"
opti_fid_path = "./OPTI_INPUT_SPECTRA/opti_total6.fid"

# output
spectra_file = "./deconvolved_data/scrf_workshop.npy"
spectra_file_names = "./deconvolved_data/scrf_workshop_names.csv"
opi_spectrum_file = "./deconvolved_data/opti_spectrum.npy"
responses_file = "./deconvolved_data/scrf_workshop.pt"
losses_file = "./deconvolved_data/losses_scrf_workshop.pt"

os.environ["DATA_DIR"] = data_dir
os.environ["OPTI_FID_PATH"] = opti_fid_path
os.environ["SPECTRA_FILE"] = spectra_file
os.environ["SPECTRA_FILE_NAMES"] = spectra_file_names
os.environ["OPI_SPECTRUM_FILE"] = opi_spectrum_file
os.environ["RESPONSES_FILE"] = responses_file
os.environ["LOSSES_FILE"] = losses_file

run()